In [ ]:
!git clone "https://github.com/fbocchi/icg.git"
from google.colab import drive # type: ignore
drive.mount("/content/drive")

In [ ]:
from os import listdir
from os.path import join
import numpy as np

from keras.applications.vgg16 import VGG16
from keras.applications.vgg16 import preprocess_input
from keras.preprocessing.image import load_img
from keras.preprocessing.image import img_to_array


def extract_features(directory, batch_size=32):

    # CNN pre-trained
    model = VGG16(
        include_top=False,
        weights="imagenet",
        pooling="avg"
    )

    print(model.summary())

    features = {}

    batch_images = []
    batch_ids = []

    def process_batch(batch_images, batch_ids):
        """Processa un batch e aggiorna il dizionario features"""

        if len(batch_images) == 0:
            return

        batch = np.array(batch_images)
        batch_features = model.predict(batch, verbose=0)

        for i, image_id in enumerate(batch_ids):
            features[image_id] = batch_features[i]

    for name in listdir(directory):

        if not name.lower().endswith((".jpg", ".jpeg", ".png")):
            continue

        filename = join(directory, name)

        # load + preprocess
        image = load_img(filename, target_size=(224, 224))
        image = img_to_array(image)
        image = preprocess_input(image)

        batch_images.append(image)
        batch_ids.append(name.split(".")[0])

        # process batch
        if len(batch_images) == batch_size:
            process_batch(batch_images, batch_ids)
            batch_images, batch_ids = [], []

    # ultimo batch
    process_batch(batch_images, batch_ids)

    return features


# =========================
# RUN FEATURE EXTRACTION
# =========================

directory = "/content/image_caption_generator/Flickr8k/Flicker8k_Dataset"

features = extract_features(directory, batch_size=32)

print(f"Extracted Features: {len(features)}")


# =========================
# SAVE (NPZ)
# =========================

# conversione dict -> formato npz
np.savez_compressed(
    "/content/drive/MyDrive/Colab Notebooks/features.npz",
    **features
)

print("Features saved to features.npz")